# 推导式：列表、字典、集合与生成器

推导式（comprehension）是 Python 中用一行代码从可迭代对象构建新容器的语法。它把「创建空容器 → 循环 → 处理元素 →（可选）筛选」压缩成一个表达式。

四种推导式长得几乎一样，区别只在最外层的括号：

- 列表推导式：`[表达式 for x in 可迭代对象]` → 得到 `list`
- 集合推导式：`{表达式 for x in 可迭代对象}` → 得到 `set`
- 字典推导式：`{键表达式: 值表达式 for x in 可迭代对象}` → 得到 `dict`
- 生成器表达式：`(表达式 for x in 可迭代对象)` → 得到生成器（惰性求值）

本 notebook 通过可运行的例子逐一理解。

## 一、列表推导式：从 for 循环演化而来

最基础的列表推导式只包含两部分：`表达式` 和 `for x in 可迭代对象`。它对每个元素求值一次表达式，把结果按顺序收集进新列表。

下面两种写法完全等价，可以对照着读。

In [1]:
squares_loop = []
for x in range(1, 6):
    squares_loop.append(x ** 2)
print(squares_loop)

squares = [x ** 2 for x in range(1, 6)]
print(squares)

[1, 4, 9, 16, 25]
[1, 4, 9, 16, 25]


表达式可以是任意复杂的表达式，只要它最终产生一个值：调用函数、格式化字符串、做条件运算都可以。

In [2]:
names = ["alice", "BOB", "Charlie"]

greetings = [f"你好，{name.capitalize()}" for name in names]
print(greetings)

lengths = [len(name) for name in names]
print(lengths)

['你好，Alice', '你好，Bob', '你好，Charlie']
[5, 3, 7]


## 二、条件：过滤用的 if 和二选一的 if-else

推导式里可以写两种「条件」，位置不同、作用不同，这是最容易混淆的地方：

- `if` 写在 **for 后面**：负责**过滤**，不满足条件的元素直接跳过，可以单独存在。
- `if-else` 写在 **for 前面**：这是条件表达式（三元表达式），每个元素都必须给出结果，只是二选一，不能省略 `else`。

一句话记忆：**for 后面的 if 决定「要不要」，for 前面的 if-else 决定「是什么」。**

In [3]:
numbers = [12, -3, 7, -20, 9, 0, -5]

# 过滤：只保留非负数（if 在 for 后面）
non_negative = [n for n in numbers if n >= 0]
print(non_negative)

# 过滤和转换可以叠加：先保留偶数，再求平方
even_squares = [n ** 2 for n in range(1, 11) if n % 2 == 0]
print(even_squares)

[12, 7, 9, 0]
[4, 16, 36, 64, 100]


In [4]:
numbers = [12, -3, 7, -20, 9, 0]

# 二选一：每个元素都要给出结果（if-else 在 for 前面）
labels = ["非负" if n >= 0 else "负数" for n in numbers]
print(labels)

# 对应的普通写法，两者结果一致
labels_loop = []
for n in numbers:
    labels_loop.append("非负" if n >= 0 else "负数")
print(labels_loop)

['非负', '负数', '非负', '负数', '非负', '非负']
['非负', '负数', '非负', '负数', '非负', '非负']


注意区分：`for` 后面只能跟不带 `else` 的 `if`；带了 `else` 就变成了条件表达式，必须整体挪到 `for` 前面。

```python
[n for n in numbers if n >= 0 else 0]   # SyntaxError：过滤位置的 if 不能带 else
[n if n >= 0 else 0 for n in numbers]   # 正确：条件表达式放在 for 前面
```

## 三、字典推导式与集合推导式

字典推导式用花括号加冒号，集合推导式用花括号不加冒号。它们同样支持 `for`、`if` 过滤和条件表达式。

In [1]:
words = ["apple", "banana", "cherry"]

# 由列表构建字典：单词 -> 长度
word_lengths = {w: len(w) for w in words}
print(word_lengths)

# 常见实用场景：反转字典
inverted = {v: k for k, v in word_lengths.items()}
print(inverted)

# 字典推导式同样可以加条件过滤
long_words = {w: len(w) for w in words if len(w) > 5}
print(long_words)

{'apple': 5, 'banana': 6, 'cherry': 6}
{5: 'apple', 6: 'cherry'}
{'banana': 6, 'cherry': 6}


In [6]:
numbers = [3, 1, 4, 1, 5, 9, 2, 6, 5, 3, 5]

# 集合自动去重，常用来「看看一列数据里有哪些不同的值」
unique = {n for n in numbers}
print(sorted(unique))

# 每个元素除以 3 的余数有哪几种
remainders = {n % 3 for n in numbers}
print(remainders)

# 集合推导式也支持条件
even_unique = {n for n in numbers if n % 2 == 0}
print(even_unique)

[1, 2, 3, 4, 5, 6, 9]
{0, 1, 2}
{2, 4, 6}


## 四、生成器表达式：惰性求值，省内存

把列表推导式的方括号换成圆括号，得到的不再是列表，而是一个**生成器**：它不会立刻计算所有元素，而是在被迭代时才逐个产出，这个特性叫**惰性求值**。

In [3]:
gen = (x ** 2 for x in range(1_000_000))
print(type(gen).__name__)

# 创建时一个元素都还没计算，迭代到哪个才算哪个
print(next(gen))
print(next(gen))

total = sum(gen)  # 对剩余元素求和
print(total)

generator
0
1
333332833333499999


In [8]:
import sys

lst = [x ** 2 for x in range(100_000)]
gen = (x ** 2 for x in range(100_000))

print(f"100000 个元素的列表占内存约 {sys.getsizeof(lst) / 1024:.0f} KB")
print(f"同样元素的生成器占内存约 {sys.getsizeof(gen) / 1024:.1f} KB")

100000 个元素的列表占内存约 782 KB
同样元素的生成器占内存约 0.2 KB


生成器表达式作为函数的**唯一参数**时，外层括号可以省略——`sum(x * x for x in ...)` 看起来没有圆括号，其实那对括号属于函数调用。

选择建议：数据要反复使用（多次遍历、切片、取长度）就用列表；数据量大且只遍历一次（求和、找最大值、逐条写入文件）就用生成器。

In [9]:
total = sum(x ** 2 for x in range(1, 101))
print(f"1~100 的平方和：{total}")

maximum = max(len(w) for w in ["apple", "banana", "cherry"])
print(f"最长单词的长度：{maximum}")

any_negative = any(n < 0 for n in [3, 1, -4])
print(f"列表中存在负数：{any_negative}")

1~100 的平方和：338350
最长单词的长度：6
列表中存在负数：True


## 五、多个 for 与嵌套结构

推导式里可以写多个 `for`，**从左到右读，就等价于普通循环从外到内嵌套**。常见用途：展开二维列表、组合多个序列（笛卡尔积）。

In [4]:
matrix = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]

# 两个 for：等价于 for row in matrix: 再 for x in row:
flat = [x for row in matrix for x in row]
print(flat)

# 对应的普通写法
flat_loop = []
for row in matrix:
    for x in row:
        flat_loop.append(x)
print(flat_loop)

[1, 2, 3, 4, 5, 6, 7, 8, 9]
[1, 2, 3, 4, 5, 6, 7, 8, 9]


In [11]:
suits = ["♠", "♥", "♦", "♣"]
ranks = ["A", "K", "Q", "J"]

# 两个 for 组合出笛卡尔积，末尾还能再接 if 过滤
red_faces = [s + r for s in suits for r in ranks if s in "♥♦"]
print(red_faces)

['♥A', '♥K', '♥Q', '♥J', '♦A', '♦K', '♦Q', '♦J']


另一种形式是**嵌套推导式**：推导式的表达式位置再放一个推导式。注意区分：

- 多个 `for` 写在同一个方括号里 → 结果被**拍平**成一层；
- 推导式里再套推导式 → 用于**保持二维结构**，比如矩阵转置。

In [12]:
matrix = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]

# 转置：外层按列号 j 遍历，内层取出每一行的第 j 个元素
transposed = [[row[j] for row in matrix] for j in range(3)]
print(transposed)

[[1, 4, 7], [2, 5, 8], [3, 6, 9]]


## 六、几个容易踩的坑

**1. 循环变量不会泄漏（Python 3）。** 推导式有自己的作用域，循环变量在推导式结束后就不存在了；普通 `for` 循环的变量则会一直留在当前作用域。

In [5]:
# 推导式：item 只存在于推导式内部
squares = [item ** 2 for item in range(3)]
print(squares)

try:
    print(item)
except NameError:
    print("item 未定义：推导式的循环变量不会泄漏到外部")

# 普通 for：循环变量在结束后依然存在
for counter in range(3):
    pass
print(f"普通 for 循环结束后 counter = {counter}")

[0, 1, 4]
item 未定义：推导式的循环变量不会泄漏到外部
普通 for 循环结束后 counter = 2


**2. 生成器只能消费一次。** 元素被取走就没了，再次迭代得到的是空结果。需要多次使用就先转成列表。

In [14]:
gen = (n for n in [1, 2, 3])

print(sum(gen))   # 第一次消费：得到 6
print(sum(gen))   # 已经耗尽：得到 0
print(list(gen))  # 同理：空列表

6
0
[]


**3. 逻辑复杂时退回普通循环。** 推导式只适合「一个表达式 + 筛选」这一种形状的逻辑。一旦需要多步计算、异常处理或多个语句，强行写成推导式会伤害可读性。

In [15]:
texts = ["  hello ", "", "WORLD x", None, "  "]

# 能跑，但读的人要停下来想一会儿
result_hard = [t.strip().title() for t in texts if isinstance(t, str) and t.strip()]

# 普通循环：同样的结果，但每一步都直白
result_clear = []
for t in texts:
    if not isinstance(t, str):
        continue
    name = t.strip().title()
    if name:
        result_clear.append(name)

print(result_hard)
print(result_clear)

['Hello', 'World X']
['Hello', 'World X']


## 七、性能小对比

推导式通常比等价的 `for + append` 略快：`append` 每次循环都要做一次属性查找，而推导式在字节码层面直接往列表里放元素。但差距不大，选推导式首先是为了可读性，性能只是顺带的好处。

In [16]:
import timeit

n = 1_000_000

t_compr = timeit.timeit("[x ** 2 for x in range(n)]", globals={"n": n}, number=10)
t_loop = timeit.timeit(
    "squares = []\nfor x in range(n):\n    squares.append(x ** 2)",
    globals={"n": n},
    number=10,
)
print(f"跑 10 次、每次 100 万个元素：")
print(f"列表推导式  {t_compr:.3f} 秒")
print(f"for+append {t_loop:.3f} 秒")

跑 10 次、每次 100 万个元素：
列表推导式  0.477 秒
for+append 0.582 秒


## 八、小练习

试着只用推导式完成下面的任务：

1. 生成 1~50 中能被 3 整除、但不能被 5 整除的数的平方列表。
2. 把 `[" a ", "", "b c", "  "]` 中每一项去除首尾空白，并丢弃空字符串。
3. 已知 `keys = ["name", "age", "city"]`、`values = ["小明", 18, "北京"]`，配合 `zip` 组成字典。
4. 用生成器表达式计算 1~10000 中所有奇数的和（不要构建列表）。

参考答案（先自己写再看）：

In [17]:
# 1. 能被 3 整除但不能被 5 整除的数的平方
result1 = [x ** 2 for x in range(1, 51) if x % 3 == 0 and x % 5 != 0]
print(result1)

# 2. 去除首尾空白并丢弃空字符串
raw = [" a ", "", "b c", "  "]
result2 = [item.strip() for item in raw if item.strip()]
print(result2)

# 3. 两个列表组成字典
keys = ["name", "age", "city"]
values = ["小明", 18, "北京"]
result3 = {k: v for k, v in zip(keys, values)}
print(result3)

# 4. 生成器表达式求奇数和
result4 = sum(n for n in range(1, 10_001) if n % 2 == 1)
print(result4)

[9, 36, 81, 144, 324, 441, 576, 729, 1089, 1296, 1521, 1764, 2304]
['a', 'b c']
{'name': '小明', 'age': 18, 'city': '北京'}
25000000


## 总结

- 四种推导式只差最外层括号：`[]` 列表、`{}` 加冒号字典、`{}` 集合、`()` 生成器。
- `for` 后面的 `if` 负责过滤（要不要），`for` 前面的 `if-else` 负责二选一（是什么）。
- 多个 `for` 从左到右读，等价于循环从外到内嵌套；嵌套推导式则用于保持二维结构。
- 生成器惰性求值、省内存，但只能消费一次；数据要反复使用就选列表。
- Python 3 中推导式的循环变量不会泄漏到外部作用域。
- 推导式的第一目标是可读：逻辑一旦变复杂（多步计算、异常处理），就退回普通循环。